In [1]:
import pandas as pd
import numpy as np 
import os
import librosa

In [2]:
DEAM_dataset = "data/DEAM"
PMEMO_dataset = "data/PMEmo2019"

In [3]:
pmemo_annotations = PMEMO_dataset + "/annotations/static_annotations.csv"
pmemo_songs = PMEMO_dataset + "/chorus"
pmemo_metadata= PMEMO_dataset + "/metadata.csv"

PMEMO_ann= pd.read_csv(pmemo_annotations, sep =",")
PMEMO_metadata = pd.read_csv(pmemo_metadata, sep =",")


In [4]:
PMEMO_ann

,musicId,Arousal(mean),Valence(mean)
0,1,0.4000,0.5750
1,4,0.2625,0.2875
2,5,0.1500,0.2000
3,6,0.5125,0.3500
4,7,0.7000,0.7250
...,...,...,...
762,993,0.8625,0.7625
763,996,0.8750,0.5625
764,997,0.7125,0.6625
765,999,0.8750,0.7750


In [5]:
mp3_df = pd.DataFrame([], columns=["musicId", "mp3_file"])

for file in os.listdir(pmemo_songs):
    data = os.path.join(pmemo_songs, file)

    audio, sr = librosa.load(data, sr = 44100)
    mp3_df = pd.concat([mp3_df, pd.DataFrame([{"musicId": file.replace(".mp3",""), "mp3_file" :audio}])])

mp3_df

,musicId,mp3_file
0,752,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
0,746,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
0,791,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
0,550,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
0,236,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
...,...,...
0,950,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
0,788,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
0,763,"[0.0, 2.6210937e-16, 2.569645e-16, 1.2659952e-..."
0,777,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [6]:
PMEMO_ann["musicId"] = PMEMO_ann["musicId"].astype(str)
PMEMO_metadata["musicId"] = PMEMO_metadata["musicId"].astype(str)
mp3_df["musicId"] = mp3_df["musicId"].astype(str)

pmemo = PMEMO_ann.join(PMEMO_metadata.set_index("musicId"), "musicId")\
                 .join(mp3_df.set_index("musicId"), "musicId")\
                 .drop(columns=["fileName","duration", "chorus_start_time", "chorus_end_time"])\
                 .rename(columns={"Valence(mean)": "Valence", "Arousal(mean)":"Arousal"})



In [7]:
pmemo

,musicId,Arousal,Valence,title,artist,album,mp3_file
0,1,0.4000,0.5750,Good Drank,2 Chainz,"Def Jam Presents: Direct Deposit, Vol. 2","[0.0, -8.106964e-16, -5.9545294e-16, 1.866628e..."
1,4,0.2625,0.2875,X Bitch (feat. Future),21 Savage,Savage Mode,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,5,0.1500,0.2000,No Heart,21 Savage,Savage Mode,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,6,0.5125,0.3500,Red Opps,21 Savage,Red Opps,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,7,0.7000,0.7250,Girls Talk Boys,5 Seconds Of Summer,Ghostbusters (Original Motion Picture Soundtrack),"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
...,...,...,...,...,...,...,...
762,993,0.8625,0.7625,Stay,Zedd,Stay,"[0.0, 2.9683367e-16, 2.2974556e-16, -6.1721226..."
763,996,0.8750,0.5625,Trouble,offaiah,Trouble,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
764,997,0.7125,0.6625,A Supplementary Story : You Never Walk Alone,방탄소년단,YOU NEVER WALK ALONE,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
765,999,0.8750,0.7750,Outro : Wings,방탄소년단,YOU NEVER WALK ALONE,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [8]:
pmemo.to_pickle("./data/PMEmo_useful")